In [1]:
import pandas as pd

# 读取 Excel 文件
df = pd.read_excel("./Rainfall data.xlsx")

# 设置列名并处理缺失值
df.columns = ["Time", "Rainfall(mm)", "Cumulative rainfall(mm)"]
df.dropna(subset=["Time"], inplace=True)

# 将 Time 列转换为 datetime 类型，并转换为 UTC 时间（减去 8 小时）
df["Time"] = pd.to_datetime(df["Time"])
df["UTC_Time"] = df["Time"] - pd.Timedelta(hours=8)

# 按照 UTC 时间排序
df.sort_values(by="UTC_Time", inplace=True)
# 筛选从 8 月 16 日开始的数据（UTC 时间）
start_date = pd.Timestamp("2023-08-16")
df_filtered = df[df["UTC_Time"] >= start_date].copy()

# 添加日期列用于分组
df_filtered["Date"] = df_filtered["UTC_Time"].dt.date

# 按日期分组，计算每日累计降雨量 val2
df_filtered["val2"] = df_filtered.groupby("Date")["Rainfall(mm)"].cumsum()

# 输出关键字段
print(df_filtered[["UTC_Time", "Rainfall(mm)", "val2"]])

# 如果需要保存为 Excel 文件，可以取消注释下面这行
# df_filtered.to_excel("daily_rainfall_cumsum.xlsx", index=False)

                UTC_Time  Rainfall(mm)  val2
96   2023-08-16 00:03:29           0.0   0.0
97   2023-08-16 00:08:29           0.0   0.0
98   2023-08-16 00:13:29           0.0   0.0
99   2023-08-16 00:18:29           0.0   0.0
100  2023-08-16 00:23:29           0.0   0.0
...                  ...           ...   ...
8616 2023-09-14 15:39:18           0.0   0.0
8617 2023-09-14 15:44:18           0.0   0.0
8618 2023-09-14 15:49:18           0.0   0.0
8619 2023-09-14 15:54:18           0.0   0.0
8620 2023-09-14 15:59:18           0.0   0.0

[8525 rows x 3 columns]


In [2]:
# 按日期分组，并取每组最后一个 val2 值（即当天的最终累计降雨量）
daily_cumsum = df_filtered.groupby('Date', as_index=False).apply(lambda x: x['val2'].iloc[-1])

# 重置索引
daily_cumsum = daily_cumsum.reset_index()

# 重命名列
daily_cumsum.columns = ['level_0', 'Date', 'Daily_Total_Rainfall(mm)']

# 删除多余的列（可选）
daily_cumsum = daily_cumsum[['Date', 'Daily_Total_Rainfall(mm)']]
dates = daily_cumsum['Date'].astype(str).tolist()
rainfalls = daily_cumsum['Daily_Total_Rainfall(mm)'].tolist()

print("dates =", dates)
print("rainfalls =", rainfalls)

dates = ['2023-08-16', '2023-08-17', '2023-08-18', '2023-08-19', '2023-08-20', '2023-08-21', '2023-08-22', '2023-08-23', '2023-08-24', '2023-08-25', '2023-08-26', '2023-08-27', '2023-08-28', '2023-08-29', '2023-08-30', '2023-08-31', '2023-09-01', '2023-09-02', '2023-09-03', '2023-09-04', '2023-09-05', '2023-09-06', '2023-09-07', '2023-09-08', '2023-09-09', '2023-09-10', '2023-09-11', '2023-09-12', '2023-09-13', '2023-09-14']
rainfalls = [0.0, 0.0, 0.0, 8.400000000000002, 0.6000000000000001, 5.6, 0.8, 0.2, 0.2, 4.800000000000001, 1.8, 1.2, 0.0, 0.6000000000000001, 1.4, 3.4000000000000004, 0.4, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 16.999999999999993, 4.800000000000001, 7.200000000000003, 0.6000000000000001, 0.4, 0.0]
